# Evidence-quality appraisal and mechanism-support analysis
**Response to Reviewer 2, Comment 2** (manuscript PROECO-D-26-03432)

This notebook reproduces, end to end, the quality appraisal of the 150 included studies and the test of whether the 15 mechanisms stay supported when stronger and weaker evidence are separated.

**Inputs (put next to this notebook):** `Thematic_Categorization_All_Papers_FIXED.xlsx` and `Resilience_dimensions.xlsx` (any file-name prefix is fine).
**Outputs (folder `output/`):** `Appendix_E_Quality_Appraisal.xlsx` (live formulas), `appendix_E_study_level.csv`, `table_3b.csv`, `fig_evidence_quality.png`, kappa templates.

**Important.** Scores come from transparent rules applied to the *extraction sheet* (Methodology, Limitations, Inputs, Mediators, Outputs, Key Findings), not from re-reading the PDFs. Treat them as a documented first pass. Correct any score through `manual_overrides.csv` (section 6) and re-run; everything downstream recalculates.

**Requirements:** `pandas`, `numpy`, `openpyxl`, `matplotlib`, `scikit-learn`.

## 1. Configuration (all thresholds are set here, before any results are seen)

In [18]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

DATA_DIR = Path('.')                     # folder with the two Excel files
OUT_DIR = Path('output'); OUT_DIR.mkdir(exist_ok=True)

THEMATIC_FILE = sorted(DATA_DIR.glob('*Thematic_Categorization_All_Papers_FIXED*.xlsx'))[-1]
RESILIENCE_FILE = sorted(DATA_DIR.glob('*Resilience dimensions*.xlsx'))[-1]

# Pre-specified cut-offs (as stated in Section 3.3.1 of the manuscript)
HIGH_MIN, MOD_MIN = 4, 3                 # score >=4 High, ==3 Moderate, <=2 Low
ROBUST_MIN, PARTIAL_MIN = 5, 2           # mechanism: >=5 high-quality studies Robust, 2-4 Partial, else Weak

# Sample-size adequacy thresholds (criterion Q2)
N_SURVEY, N_EXPERIMENT, N_ARCHIVAL, N_META, N_QUAL = 150, 100, 500, 50, 10

SEED = 42                                # random seed for the second-coder sample
print('Thematic file  :', THEMATIC_FILE.name)
print('Resilience file:', RESILIENCE_FILE.name)

Thematic file  : Thematic_Categorization_All_Papers_FIXED.xlsx
Resilience file: Resilience dimensions.xlsx


## 2. Load and merge the two files (one row per study)

In [19]:
def key(s):
    """Normalised title key used to link the two files."""
    return re.sub(r'[^a-z0-9]', '', str(s).lower().replace('.pdf', ''))

# Titles that differ between the two files for the SAME paper (verify manually; see data checks in section 9)
ALIAS = {
    key('Blockchain technology adaptation and organizational inertia: moderating role between knowledge '
        'management processes and supply chain resilience'):
    key('Blockchain technology adaptation and supply chain performance (Emerald – IJPDLM)'),
}
SHEETS = {'Data_Analytics_Papers': 'Analytics', 'Blockchain_Papers': 'Blockchain', 'Combined_Tech_Papers': 'Integrated'}
TEXT_COLS = ['Author(s), Year', 'Methodology', 'Limitations', 'Inputs', 'Mediators/Moderators', 'Outputs',
             'Key Findings', 'Resilience Dimensions', 'Technology Type']

T = pd.read_excel(THEMATIC_FILE, sheet_name=None)
R = pd.read_excel(RESILIENCE_FILE, sheet_name=None)

parts = []
for sh, stream in SHEETS.items():
    r, t = R[sh].copy(), T[sh].copy()
    r['key'] = r['File Name'].map(lambda x: ALIAS.get(key(x), key(x)))
    t['key'] = t['File Name'].map(key)
    m = r.merge(t.drop_duplicates('key'), on='key', how='left', suffixes=('', '_t'))
    m['Stream'] = stream
    m['StudyID'] = [f'{stream[0]}{i + 1:03d}' for i in range(len(m))]      # A001.., B001.., I001..
    parts.append(m)
df = pd.concat(parts, ignore_index=True)

# A row whose every field repeats the title is a corrupted extraction: treat as missing
bad = df['Methodology'].astype(str).str.lower().str.strip() == df['File Name'].astype(str).str.lower().str.strip()
df.loc[bad, TEXT_COLS] = np.nan

assert len(df) == 150, f'Expected 150 studies, found {len(df)}'
print(df.groupby('Stream').size().to_dict(), '| studies with no methodology text:', int(df['Methodology'].isna().sum()))

{'Analytics': 83, 'Blockchain': 63, 'Integrated': 4} | studies with no methodology text: 0


## 3. Appraisal rules
Five binary criteria (1 = met, 0 = not met or not reported), summed to 0-5.

| ID | Criterion | Scored 1 when |
|---|---|---|
| Q1 | Design strength | Design goes beyond a single-wave, single-source survey (multi-wave/multi-respondent, mixed or multi-source, experimental, quasi-experimental/archival, meta-analytic, multi-case or longitudinal qualitative) |
| Q2 | Sample adequacy | Surveys n>=150; experiments n>=100; archival/panel >=500 obs.; meta-analysis >=50 effect sizes; qualitative >=10 participants or multiple cases |
| Q3 | Measurement / data credibility | Validation steps (EFA/CFA, pilot, Delphi, multilevel CFA), or objective/archival/secondary data, or experimental manipulation, or documented qualitative analysis |
| Q4 | Bias control | Multi-respondent, two-wave, matched, secondary data, PSM, robustness checks, ML validation, triangulation, or identification design; 0 for single-respondent surveys |
| Q5 | Analytical rigour | Bootstrapped mediation/moderation, robustness checks, fsQCA, causal-identification methods, multilevel or meta-analytic methods |

Tiers: **High** 4-5, **Moderate** 3, **Low** 0-2.

In [20]:
def has(txt, pats):
    return any(re.search(p, txt, flags=re.I) for p in pats)

# ---- sample size -------------------------------------------------------------------------
N_OVERRIDE = {                         # text fragment -> n (cases where several numbers appear)
    '516 distributed': 283, '1000 firms contacted': 188, '384 collected': 260, '380 distributed': 142,
    '415 establishments': 336, '200 distributed': 142, 'random sample 3000': 438, 'emailed to 760': 257,
    '129 responses from 39': 129, 'interviews (n=10)': 219, 'n=117; n=197': 197, '44 experts': 229,
    '373 organizations': 373, '281 surveys': 281, 'sample details tbd': None,
}
NUM = r'(\d[\d,]*)'
def get_n(method_text):
    m = str(method_text).replace('\n', ' '); low = m.lower()
    for frag, val in N_OVERRIDE.items():
        if frag in low:
            return val
    pats = [r'\bn\s*[-=≈:]\s*~?' + NUM,                                       # n=164, n-362
            NUM + r'\s+(?:usable|valid|complete)',                              # 438 valid
            NUM + r'\s+match(?:ed)?-?\s?pair(?:ed)?\s+surveys',                 # 121 match-paired surveys
            r'\(?' + NUM + r'\s*\)?\s*(?:responses|respondents|managers|professionals|participants|firms|'
            r'experts|employees|observations|firm-year|effect sizes|surveys)',
            r'\bN\s*=\s*' + NUM,
            r'\(\s*' + NUM + r'\s*\)']                                          # Survey (187 )
    for p in pats:                                                              # first pattern family that hits wins
        c = [int(x.replace(',', '')) for x in re.findall(p, m, flags=re.I)]
        if c:
            return max(c)
    return None

# ---- design classification ---------------------------------------------------------------
DESIGNS = ['Cross-sectional single-source survey', 'Multi-wave / multi-respondent survey', 'Mixed / multi-source',
           'Qualitative / case study', 'Quasi-experimental / archival / panel',
           'Experimental (vignette / between-subjects)', 'Meta-analysis']
STRONG = {'Meta-analysis', 'Quasi-experimental / archival / panel', 'Experimental (vignette / between-subjects)',
          'Mixed / multi-source', 'Multi-wave / multi-respondent survey'}

def classify_design(method_text):
    t = str(method_text).lower()
    if has(t, [r'meta-analysis']): return 'Meta-analysis'
    if has(t, [r'quasi-experiment', r'difference in difference', r'\bdid\b', r'event study', r'panel econometrics',
               r'fixed effects', r'archival', r'system gmm']): return 'Quasi-experimental / archival / panel'
    if has(t, [r'experiment', r'vignette']): return 'Experimental (vignette / between-subjects)'
    survey = has(t, [r'survey', r'questionnaire', r'\bsem\b', r'pls-sem', r'regression', r'hypothesis testing', r'n\s*[=-]', r'process'])
    qual = has(t, [r'interview', r'case stud', r'qualitative', r'tism', r'text analysis', r'thematic', r'expert interview'])
    second_source = has(t, [r'secondary data', r'big data analysis', r'literature synthesis'])
    multiwave = has(t, [r'two-wave', r'multi-respondent', r'two-respondent', r'match(?:ed)?-?\s?pair', r'dual respondent'])
    if survey and (qual or second_source): return 'Mixed / multi-source'     # needs >=2 data-collection modes
    if qual and not survey: return 'Qualitative / case study'
    if multiwave: return 'Multi-wave / multi-respondent survey'
    if survey: return 'Cross-sectional single-source survey'
    return 'Not classifiable'

# ---- criteria Q1-Q5 ----------------------------------------------------------------------
def appraise(row):
    if pd.isna(row['Methodology']):
        return pd.Series({'Design': 'Not appraised', 'n': np.nan, 'Q1': np.nan, 'Q2': np.nan, 'Q3': np.nan,
                          'Q4': np.nan, 'Q5': np.nan, 'Rule_notes': 'no methodology text'})
    m, L = str(row['Methodology']), str(row['Limitations']); mt, Lt = m.lower(), L.lower()
    des, n = classify_design(m), get_n(m); notes = []
    q1 = int(des in STRONG)
    if des == 'Qualitative / case study':
        q1 = int(has(mt, [r'multiple case', r'longitudinal', r'four', r'text analysis', r'case stud']))
    nn = n or 0
    if des == 'Meta-analysis':                              q2 = int(nn >= N_META)
    elif des == 'Quasi-experimental / archival / panel':    q2 = int(nn >= N_ARCHIVAL)
    elif des.startswith('Experimental'):                    q2 = int(nn >= N_EXPERIMENT)
    elif des == 'Qualitative / case study':                 q2 = int(has(mt, [r'multiple case', r'four']) or nn >= N_QUAL)
    else:                                                   q2 = int(nn >= N_SURVEY)
    q3 = int(des in ('Meta-analysis', 'Quasi-experimental / archival / panel') or des.startswith('Experimental') or
             has(mt, [r'\befa\b', r'\bcfa\b', r'delphi', r'pilot', r'multilevel', r'validat', r'thematic', r'text analysis',
                      r'secondary data', r'triangulat', r'follow-up interviews']))
    q4 = int(des in ('Meta-analysis', 'Mixed / multi-source') or
             has(mt, [r'two-wave', r'multi-respondent', r'two-respondent', r'match(?:ed)?-?\s?pair', r'secondary data', r'\bpsm\b',
                      r'robustness', r'machine learning', r'triangulat', r'follow-up interviews', r'experiment', r'vignette',
                      r'difference in difference', r'event study', r'gmm', r'fixed effects', r'archival']))
    if has(Lt, [r'single-respondent', r'single-informant']) and des == 'Cross-sectional single-source survey':
        q4 = 0
    q5 = int(des in ('Meta-analysis', 'Quasi-experimental / archival / panel') or
             has(mt, [r'mediation', r'moderat', r'bootstrap', r'process', r'robustness', r'fsqca', r'machine learning', r'\bpsm\b',
                      r'difference in difference', r'gmm', r'fixed effects', r'meta-sem', r'meta-regression', r'multilevel',
                      r'event study', r'anfis', r'regression analysis', r'hypothesis testing']))
    if des == 'Qualitative / case study':
        q5 = int(has(mt, [r'text analysis', r'thematic', r'multiple', r'longitudinal', r'four']))
    if n is None and des != 'Qualitative / case study': notes.append('sample size not in extraction sheet (Q2 scored 0)')
    if 'tbd' in Lt or 'tbd' in mt: notes.append('TBD in extraction sheet')
    return pd.Series({'Design': des, 'n': n, 'Q1': q1, 'Q2': q2, 'Q3': q3, 'Q4': q4, 'Q5': q5, 'Rule_notes': '; '.join(notes)})

## 4. Mechanism coding (F1-F15)
A study is coded 1 for a mechanism when the mechanism's keywords appear in its extracted *Inputs, Mediators/Moderators, Outputs, Key Findings, Resilience Dimensions or Technology Type*. This records that the study **examines** the mechanism, not that its effect was significant or positive. Keyword recall is imperfect (governance terms are probably under-counted), so review the counts and override where needed.

In [21]:
MECH = {
 'F1':  ('Transparency',                  [r'transparen']),
 'F2':  ('Traceability',                  [r'traceab', r'provenance']),
 'F3':  ('Data integrity',                [r'integrity', r'data quality', r'data security', r'tamper', r'immutab']),
 'F4':  ('Information sharing',           [r'information sharing', r'data sharing', r'knowledge sharing', r'information integration', r'interoperab', r'information exchange']),
 'F5':  ('Trust',                         [r'\btrust']),
 'F6':  ('Risk sensing',                  [r'sensing', r'early warning', r'anticipat', r'risk identif', r'risk detect', r'risk assess']),
 'F7':  ('Predictive capability',         [r'predictive', r'forecast', r'prediction']),
 'F8':  ('Analytics capability',          [r'analytics', r'\bbda', r'big data', r'machine learning', r'artificial intelligence']),
 'F9':  ('Supply chain visibility',       [r'visibility']),
 'F10': ('Decision intelligence',         [r'decision']),
 'F11': ('Collaborative decision-making', [r'collaborat', r'coordinat', r'joint decision']),
 'F12': ('Agile response',                [r'agil', r'responsiveness', r'flexib', r'adaptab']),
 'F13': ('Recovery capability',           [r'recover', r'restor', r'reactive']),
 'F14': ('Continuity planning',           [r'continuity', r'preparedness', r'readiness', r'contingenc']),
 'F15': ('Organizational learning',       [r'learning', r'absorptive', r'ambidexter']),
}
MECH_FIELDS = ['Inputs', 'Mediators/Moderators', 'Outputs', 'Key Findings', 'Resilience Dimensions', 'Technology Type']
F_COLS = list(MECH)

def mech_flags(row):
    txt = ' '.join(str(row[c]) for c in MECH_FIELDS if pd.notna(row[c])).lower()
    return pd.Series({k: (int(has(txt, p)) if txt.strip() else np.nan) for k, (_, p) in MECH.items()})

def null_flag(row):
    return int(has(str(row['Key Findings']).lower(), [r'non-significant', r'not significant', r'no significant', r'negatively',
                                                       r'do not moderate', r'without a significant', r'does not', r'mixed']))

## 5. Apply the rules to all 150 studies

In [22]:
A = df.apply(appraise, axis=1)
F = df.apply(mech_flags, axis=1)

d = pd.concat([df[['StudyID', 'Stream', 'Assigned Theme', 'Author(s), Year', 'File Name', 'Industry/Sector',
                   'Theoretical Lens', 'Resilience Dimension (Short)']].rename(columns={'Assigned Theme': 'Theme'}), A, F], axis=1)
d['Theme'] = d['Theme'].astype(str).str.split(':').str[0]
d['Study'] = d['Author(s), Year'].fillna(d['File Name'].str[:80])
d['Null_or_mixed_finding'] = df.apply(null_flag, axis=1)
d['Check_flag'] = ''
d.head(3)

,StudyID,Stream,Theme,"Author(s), Year",File Name,Industry/Sector,Theoretical Lens,Resilience Dimension (Short),Design,n,...,F9,F10,F11,F12,F13,F14,F15,Study,Null_or_mixed_finding,Check_flag
0,A001,Analytics,Theme 3,"Fernando & Ikhsan, 2023",A Data-Driven Supply Chain: Marketing Data Sha...,Manufacturing,Not stated,Adaptability,Cross-sectional single-source survey,375.0,...,0,0,0,0,0,0,0,"Fernando & Ikhsan, 2023",0,
1,A002,Analytics,Theme 4,"Xu & Liu, 2024",Achieving manufacturing supply chain resilienc...,Manufacturing,Cognitive-behavioral theory,Resilience,Multi-wave / multi-respondent survey,164.0,...,0,0,0,0,0,0,1,"Xu & Liu, 2024",1,
2,A003,Analytics,Theme 3,"Trabucco & De Giovanni, 2021",Achieving Resilience and Business Sustainabili...,Mixed sectors,Not stated,Viability,Cross-sectional single-source survey,119.0,...,0,0,1,0,1,0,1,"Trabucco & De Giovanni, 2021",1,


## 6. Manual overrides (optional)
Create `manual_overrides.csv` with columns `StudyID, field, value` (field = `n`, `Design`, `Q1`...`Q5`, or `F1`...`F15`). Overrides are applied after the rules, so every correction you make after reading a full text is documented and reproducible. A template listing the studies whose sample size is missing is written to `output/manual_overrides_template.csv`.

In [23]:
ov_path = DATA_DIR / 'manual_overrides.csv'
if ov_path.exists():
    ov = pd.read_csv(ov_path)
    for _, o in ov.iterrows():
        v = pd.to_numeric(pd.Series([o['value']]), errors='coerce').iloc[0]
        d.loc[d['StudyID'] == o['StudyID'], o['field']] = o['value'] if pd.isna(v) else v
    print(f'Applied {len(ov)} manual overrides from {ov_path.name}')
else:
    print('No manual_overrides.csv found: rule-based scores are used as they are.')

q = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5']
d['Score'] = d[q].sum(axis=1, min_count=1)
d['Tier'] = d['Score'].map(lambda s: 'Not appraised' if pd.isna(s) else
                           ('High' if s >= HIGH_MIN else 'Moderate' if s >= MOD_MIN else 'Low'))

tmpl = d[d['n'].isna() & ~d['Design'].isin(['Qualitative / case study'])][['StudyID', 'Study', 'Design']].copy()
tmpl.insert(2, 'field', 'n'); tmpl['value'] = ''
tmpl.to_csv(OUT_DIR / 'manual_overrides_template.csv', index=False)
print(f'{len(tmpl)} studies have no sample size in the extraction sheet -> see output/manual_overrides_template.csv')

No manual_overrides.csv found: rule-based scores are used as they are.
14 studies have no sample size in the extraction sheet -> see output/manual_overrides_template.csv


## 7. Results

### 7.1 Quality tiers by stream (Table S1)

In [24]:
app = d[d['Tier'] != 'Not appraised']
t1 = pd.crosstab(d['Stream'], d['Tier']).reindex(columns=['High', 'Moderate', 'Low', 'Not appraised'], fill_value=0)
t1.loc['Total'] = t1.sum()
t1['Appraised'] = t1[['High', 'Moderate', 'Low']].sum(axis=1)
for c in ['High', 'Moderate', 'Low']:
    t1[f'% {c}'] = (100 * t1[c] / t1['Appraised']).round(1)
t1['Mean score'] = [round(app.loc[app['Stream'] == s, 'Score'].mean(), 2) for s in t1.index[:-1]] + [round(app['Score'].mean(), 2)]
t1

Tier,High,Moderate,Low,Not appraised,Appraised,% High,% Moderate,% Low,Mean score
Stream,,,,,,,,,
Analytics,12,4,67,0,83,14.5,4.8,80.7,1.64
Blockchain,4,6,53,0,63,6.3,9.5,84.1,1.29
Integrated,1,0,3,0,4,25.0,0.0,75.0,2.50
Total,17,10,123,0,150,11.3,6.7,82.0,1.51


### 7.2 Design profile and criteria met (Tables S2 and S3)

In [25]:
t2 = app['Design'].value_counts().reindex(DESIGNS, fill_value=0).to_frame('Studies')
t2['% of appraised'] = (100 * t2['Studies'] / len(app)).round(1)
display(t2)
t3 = (100 * app[q].mean()).round(1).rename({'Q1': 'Q1 Design strength', 'Q2': 'Q2 Sample adequacy', 'Q3': 'Q3 Measurement/data credibility',
                                             'Q4': 'Q4 Bias control', 'Q5': 'Q5 Analytical rigour'}).to_frame('% of studies meeting criterion')
display(t3)
print('Studies flagged with null/negative/mixed findings in the extraction sheet:', int(d['Null_or_mixed_finding'].sum()))

,Studies,% of appraised
Design,,
Cross-sectional single-source survey,118,78.7
Multi-wave / multi-respondent survey,7,4.7
Mixed / multi-source,10,6.7
Qualitative / case study,7,4.7
Quasi-experimental / archival / panel,5,3.3
Experimental (vignette / between-subjects),2,1.3
Meta-analysis,1,0.7


,% of studies meeting criterion
Q1 Design strength,19.3
Q2 Sample adequacy,78.7
Q3 Measurement/data credibility,12.7
Q4 Bias control,18.0
Q5 Analytical rigour,22.7


Studies flagged with null/negative/mixed findings in the extraction sheet: 19


### 7.3 Table 3b: mechanism support by evidence quality (primary and sensitivity)

In [26]:
def support(k, robust=ROBUST_MIN, partial=PARTIAL_MIN):
    return 'Robust' if k >= robust else 'Partial' if k >= partial else 'Weak'

def table_3b(frame, robust=ROBUST_MIN, partial=PARTIAL_MIN):
    rows = []
    for f, (name, _) in MECH.items():
        s = frame[frame[f] == 1]
        hi, mo, lo = (s['Tier'] == 'High').sum(), (s['Tier'] == 'Moderate').sum(), (s['Tier'] == 'Low').sum()
        rows.append({'Factor': f, 'Mechanism': name, 'All studies': len(s), 'High': hi, 'Moderate': mo, 'Low': lo,
                     'Support (high only; pre-specified)': support(hi, robust, partial),
                     'High + Moderate': hi + mo, 'Support (sensitivity: high + moderate)': support(hi + mo, robust, partial),
                     '% High + Moderate': round(100 * (hi + mo) / len(s), 1) if len(s) else 0})
    return pd.DataFrame(rows)

t3b = table_3b(app)
t3b.to_csv(OUT_DIR / 'table_3b.csv', index=False)
t3b

,Factor,Mechanism,All studies,High,Moderate,Low,Support (high only; pre-specified),High + Moderate,Support (sensitivity: high + moderate),% High + Moderate
0,F1,Transparency,22,1,1,20,Weak,2,Partial,9.1
1,F2,Traceability,11,0,1,10,Weak,1,Weak,9.1
2,F3,Data integrity,3,0,0,3,Weak,0,Weak,0.0
3,F4,Information sharing,11,2,0,9,Partial,2,Partial,18.2
4,F5,Trust,15,2,1,12,Partial,3,Partial,20.0
5,F6,Risk sensing,6,0,1,5,Weak,1,Weak,16.7
6,F7,Predictive capability,6,3,1,2,Partial,4,Partial,66.7
7,F8,Analytics capability,89,13,5,71,Robust,18,Robust,20.2
8,F9,Supply chain visibility,19,4,2,13,Partial,6,Robust,31.6
9,F10,Decision intelligence,7,1,0,6,Weak,1,Weak,14.3


In [27]:
for lab, col in [('pre-specified (high-quality studies only)', 'Support (high only; pre-specified)'),
                 ('sensitivity (high + moderate)', 'Support (sensitivity: high + moderate)')]:
    print(f'\nSupport classification, {lab}:')
    for cls in ['Robust', 'Partial', 'Weak']:
        names = t3b.loc[t3b[col] == cls, 'Mechanism'].tolist()
        print(f'  {cls:8s} ({len(names):2d}): ' + ', '.join(names))


Support classification, pre-specified (high-quality studies only):
  Robust   ( 2): Analytics capability, Agile response
  Partial  ( 6): Information sharing, Trust, Predictive capability, Supply chain visibility, Collaborative decision-making, Recovery capability
  Weak     ( 7): Transparency, Traceability, Data integrity, Risk sensing, Decision intelligence, Continuity planning, Organizational learning

Support classification, sensitivity (high + moderate):
  Robust   ( 4): Analytics capability, Supply chain visibility, Agile response, Recovery capability
  Partial  ( 5): Transparency, Information sharing, Trust, Predictive capability, Collaborative decision-making
  Weak     ( 6): Traceability, Data integrity, Risk sensing, Decision intelligence, Continuity planning, Organizational learning


### 7.4 Robustness of the classification to the cut-offs
The cut-offs were fixed in advance. This table shows how many mechanisms fall in each class under alternative cut-offs, so readers can see how much the conclusion depends on them.

In [28]:
rows = []
for hmin in (3, 4, 5):
    tier = app['Score'].map(lambda s: 'High' if s >= hmin else 'Moderate' if s >= hmin - 1 else 'Low')
    tmp = app.assign(Tier=tier)
    for rmin, pmin in ((3, 2), (5, 2), (5, 3)):
        tb = table_3b(tmp, rmin, pmin)
        c = tb['Support (high only; pre-specified)'].value_counts()
        rows.append({'High tier if score >=': hmin, 'High-quality studies in corpus': int((tier == 'High').sum()),
                     'Robust if >=': rmin, 'Partial if >=': pmin,
                     'Robust': int(c.get('Robust', 0)), 'Partial': int(c.get('Partial', 0)), 'Weak': int(c.get('Weak', 0))})
sweep = pd.DataFrame(rows)
sweep.to_csv(OUT_DIR / 'threshold_sweep.csv', index=False)
sweep

,High tier if score >=,High-quality studies in corpus,Robust if >=,Partial if >=,Robust,Partial,Weak
0,3,27,3,2,7,2,6
1,3,27,5,2,4,5,6
2,3,27,5,3,4,3,8
3,4,17,3,2,5,3,7
4,4,17,5,2,2,6,7
5,4,17,5,3,2,3,10
6,5,6,3,2,2,1,12
7,5,6,5,2,1,2,12
8,5,6,5,3,1,1,13


### 7.5 Figure

In [29]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
colors = {'High': '#2E7D32', 'Moderate': '#F9A825', 'Low': '#C62828'}
fig, ax = plt.subplots(1, 2, figsize=(13, 5.5), gridspec_kw={'width_ratios': [1, 2]})

streams = ['Analytics', 'Blockchain', 'Integrated']
bottom = np.zeros(3)
for tier in ['High', 'Moderate', 'Low']:
    vals = np.array([(app[(app['Stream'] == s)]['Tier'] == tier).mean() * 100 for s in streams])
    ax[0].bar(streams, vals, bottom=bottom, color=colors[tier], label=tier); bottom += vals
ax[0].set_ylabel('% of appraised studies'); ax[0].set_title('(a) Evidence quality by stream'); ax[0].legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.07), ncol=3)

order = t3b.sort_values('All studies')
left = np.zeros(len(order))
for tier in ['High', 'Moderate', 'Low']:
    ax[1].barh(order['Factor'] + ' ' + order['Mechanism'], order[tier], left=left, color=colors[tier], label=tier); left += order[tier].values
ax[1].set_xlabel('Studies examining the mechanism'); ax[1].set_title('(b) Mechanism support by evidence quality')
for s in ('top', 'right'):
    ax[0].spines[s].set_visible(False); ax[1].spines[s].set_visible(False)
plt.tight_layout(); plt.savefig(OUT_DIR / 'fig_evidence_quality.png', dpi=300); plt.show()

C:\Users\1886199\AppData\Local\Temp\ipykernel_25544\3068211569.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(OUT_DIR / 'fig_evidence_quality.png', dpi=300); plt.show()


In [30]:
txt_all = (df[['File Name', 'Technology Type', 'Inputs', 'Mediators/Moderators', 'Outputs', 'Key Findings', 'Methodology']]
    .apply(lambda r: ' '.join(map(str, r.values)), axis=1).str.lower())
d['Blockchain_explicit'] = txt_all.str.contains(r'blockchain|distributed ledger|smart contract|\bdlt\b')
d['Title_resilience'] = df['File Name'].astype(str).str.lower().str.contains(r'resilien|disruption')
d['Title_traceability'] = df['File Name'].astype(str).str.lower().str.contains(r'traceab')

# (1) technology focus
bc = d[d['Stream'].isin(['Blockchain', 'Integrated'])]
print(f"(1) Blockchain/integrated studies that explicitly examine blockchain/DLT/smart contracts: "
      f"{int(bc['Blockchain_explicit'].sum())} of {len(bc)}")
print('    Not explicit:', bc.loc[~bc['Blockchain_explicit'], 'Study'].tolist())
print(f"    Blockchain-stream studies with 'traceability' in the title: {int(bc['Title_traceability'].sum())}")

# (2) resilience-anchored subset (title names resilience or disruption)
core = d[d['Title_resilience']]
print(f"\n(2) Studies whose title names resilience/disruption: {len(core)} of {len(d)} ({100*len(core)/len(d):.1f}%)")
print(d.groupby('Stream')['Title_resilience'].agg(['sum', 'count']).rename(columns={'sum': 'title_resilience', 'count': 'studies'}))

LINKED = {'F2': 'Traceability (search term)', 'F9': 'Visibility (eligibility)', 'F11': 'Collaboration (eligibility)',
          'F12': 'Agility / flexibility (eligibility)', 'F13': 'Recovery (search + eligibility)', 'F14': 'Continuity / preparedness (search + eligibility)'}
rows = []
for f, (name, _) in MECH.items():
    s_all, s_core, s_rest = d[d[f] == 1], core[core[f] == 1], d[(d[f] == 1) & ~d['Title_resilience']]
    rows.append({'Factor': f, 'Mechanism': name, 'Shares a word with search/eligibility terms': 'Yes' if f in LINKED else 'No',
                 'Studies (all)': len(s_all), 'Studies in resilience-titled subset': len(s_core),
                 '% in subset': round(100 * len(s_core) / len(s_all), 1) if len(s_all) else 0,
                 'High-quality studies in subset': int((s_core['Tier'] == 'High').sum())})
circ = pd.DataFrame(rows)
circ.to_csv(OUT_DIR / 'circularity_check.csv', index=False)
print('\n(3) Mechanism evidence inside the resilience-titled subset:')
display(circ)
print('\nMechanisms examined by at least one resilience-titled study:', int((circ['Studies in resilience-titled subset'] > 0).sum()), 'of 15')
print('Mechanisms examined by at least three resilience-titled studies:', int((circ['Studies in resilience-titled subset'] >= 3).sum()), 'of 15')
g = circ.groupby('Shares a word with search/eligibility terms')[['Studies (all)', 'Studies in resilience-titled subset']].sum()
g['mean % in subset'] = circ.groupby('Shares a word with search/eligibility terms')['% in subset'].mean().round(1)
display(g)

(1) Blockchain/integrated studies that explicitly examine blockchain/DLT/smart contracts: 67 of 67
    Not explicit: []
    Blockchain-stream studies with 'traceability' in the title: 0

(2) Studies whose title names resilience/disruption: 97 of 150 (64.7%)
            title_resilience  studies
Stream                               
Analytics                 61       83
Blockchain                35       63
Integrated                 1        4

(3) Mechanism evidence inside the resilience-titled subset:


,Factor,Mechanism,Shares a word with search/eligibility terms,Studies (all),Studies in resilience-titled subset,% in subset,High-quality studies in subset
0,F1,Transparency,No,22,13,59.1,1
1,F2,Traceability,Yes,11,6,54.5,0
2,F3,Data integrity,No,3,3,100.0,0
3,F4,Information sharing,No,11,6,54.5,2
4,F5,Trust,No,15,8,53.3,1
5,F6,Risk sensing,No,6,6,100.0,0
6,F7,Predictive capability,No,6,4,66.7,2
7,F8,Analytics capability,No,89,64,71.9,10
8,F9,Supply chain visibility,Yes,19,9,47.4,2
9,F10,Decision intelligence,No,7,4,57.1,1



Mechanisms examined by at least one resilience-titled study: 15 of 15
Mechanisms examined by at least three resilience-titled studies: 15 of 15


,Studies (all),Studies in resilience-titled subset,mean % in subset
Shares a word with search/eligibility terms,,,
No,173,117,69.7
Yes,156,100,66.6


## 8. Second-coder reliability (to report kappa)
1. Run the next cell: it draws a reproducible random sample of 30 studies (seed 42) and writes `output/second_coder_template.csv` (study names and blank Q1-Q5 columns, **without** the first coder's scores).
2. A second author scores those 30 studies independently from the full texts and saves the file as `second_coder.csv` next to this notebook.
3. Re-run the notebook: Cohen's kappa and percent agreement are computed per criterion and for the tier.

In [31]:
from sklearn.metrics import cohen_kappa_score

sample30 = app.sample(30, random_state=SEED).sort_values('StudyID')
tmpl2 = sample30[['StudyID', 'Study']].copy()
for c in q: tmpl2[c] = ''
tmpl2.to_csv(OUT_DIR / 'second_coder_template.csv', index=False)

sc_path = DATA_DIR / 'second_coder.csv'
if sc_path.exists():
    sc = pd.read_csv(sc_path).set_index('StudyID')
    first = d.set_index('StudyID').loc[sc.index]
    rows = []
    for c in q:
        a, b = first[c].astype(int), sc[c].astype(int)
        rows.append({'Criterion': c, 'Agreement %': round(100 * (a == b).mean(), 1), "Cohen's kappa": round(cohen_kappa_score(a, b), 2)})
    t_first = first['Score'].map(lambda s: 'High' if s >= HIGH_MIN else 'Moderate' if s >= MOD_MIN else 'Low')
    t_sec = sc[q].astype(int).sum(axis=1).map(lambda s: 'High' if s >= HIGH_MIN else 'Moderate' if s >= MOD_MIN else 'Low')
    rows.append({'Criterion': 'Tier', 'Agreement %': round(100 * (t_first == t_sec).mean(), 1),
                 "Cohen's kappa": round(cohen_kappa_score(t_first, t_sec), 2)})
    display(pd.DataFrame(rows))
else:
    print('second_coder.csv not found: template written to output/second_coder_template.csv')

second_coder.csv not found: template written to output/second_coder_template.csv


## 9. Data-quality checks to resolve before submission

In [32]:
chk = []
dup = d[d.duplicated(['Stream', 'Author(s), Year'], keep=False) & d['Author(s), Year'].notna()]
for _, r in dup.iterrows():
    chk.append((r['StudyID'], r['Study'], 'Same author-year appears more than once in this stream: confirm they are distinct papers / correct the year'))
for k_, v_ in ALIAS.items():
    chk.append(('B-link', 'organizational-inertia paper', 'Linked across files by title alias: confirm the two titles are the same paper'))
for _, r in d[d['n'].isna() & ~d['Design'].isin(['Qualitative / case study'])].iterrows():
    chk.append((r['StudyID'], r['Study'], 'Sample size missing in extraction sheet: add it (Q2 is scored 0 until then)'))
for _, r in d[d['Rule_notes'].str.contains('TBD', na=False)].iterrows():
    chk.append((r['StudyID'], r['Study'], 'TBD in extraction sheet: complete from full text'))
checks = pd.DataFrame(chk, columns=['StudyID', 'Study', 'Action'])
checks.to_csv(OUT_DIR / 'data_checks.csv', index=False)
print(len(checks), 'items flagged; see output/data_checks.csv')
checks.head(40)

20 items flagged; see output/data_checks.csv


,StudyID,Study,Action
0,B015,"Pattanayak et al., 2024",Same author-year appears more than once in thi...
1,B017,"Pattanayak et al., 2024",Same author-year appears more than once in thi...
2,B-link,organizational-inertia paper,Linked across files by title alias: confirm th...
3,A063,"Shokoohyar, S.; Shokouhyar, S.; Hakimioun, S. ...",Sample size missing in extraction sheet: add i...
4,A066,"Abdullah, Naeem & Aslam, 2024",Sample size missing in extraction sheet: add i...
5,B001,"Hajian, Rezaeinejad, Rayman, & Khorsandroo (2025)",Sample size missing in extraction sheet: add i...
6,B005,Ganesh Kumar R; Dinesh Kumar S; C. Anirvinna; ...,Sample size missing in extraction sheet: add i...
7,B006,"Haidi Zhou; Qiang Wang; Xiande Zhao, 2025",Sample size missing in extraction sheet: add i...
8,B008,Maciel M. Queiroz; Samuel Fosso Wamba; Marc De...,Sample size missing in extraction sheet: add i...
9,B016,"Dubey et al., 2020",Sample size missing in extraction sheet: add i...


## 10. Export: study-level CSV and the Excel appendix (live formulas)

In [33]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter as L

out_cols = ['StudyID', 'Stream', 'Theme', 'Study', 'Industry/Sector', 'Theoretical Lens', 'Resilience Dimension (Short)', 'Design', 'n'] + q + ['Score', 'Tier'] + F_COLS + ['Null_or_mixed_finding', 'Rule_notes']
d[out_cols].to_csv(OUT_DIR / 'appendix_E_study_level.csv', index=False)

def build_workbook(d, path):
    AR, AB = Font(name='Arial', size=10), Font(name='Arial', size=10, bold=True)
    BLUE, HDR = Font(name='Arial', size=10, color='0000FF'), PatternFill('solid', fgColor='D9E1F2')
    wb = Workbook(); S = wb.active; S.title = 'Summary'; E = wb.create_sheet('Appendix_E')
    N = len(d); first, last = 2, N + 1
    hdr = ['ID', 'Stream', 'Theme', 'Study', 'Sector', 'Theoretical lens', 'Resilience dimension', 'Design', 'n',
           'Q1 Design strength', 'Q2 Sample adequacy', 'Q3 Measurement/data', 'Q4 Bias control', 'Q5 Analysis', 'Score (0-5)', 'Tier'] + \
          [f'{k} {MECH[k][0]}' for k in MECH] + ['Null/mixed finding flag', 'Rule notes']
    for j, h in enumerate(hdr, 1):
        c = E.cell(1, j, h); c.font = AB; c.fill = HDR; c.alignment = Alignment(wrap_text=True, vertical='center')
    def val(v): return None if pd.isna(v) else (int(v) if isinstance(v, (int, float, np.integer, np.floating)) else v)
    for i, (_, r) in enumerate(d.iterrows(), 2):
        base = [r['StudyID'], r['Stream'], r['Theme'], r['Study'], r['Industry/Sector'], r['Theoretical Lens'],
                r['Resilience Dimension (Short)'], r['Design'], val(r['n'])]
        for j, v in enumerate(base, 1): E.cell(i, j, v).font = AR
        for j, c_ in enumerate(q, 10): E.cell(i, j, val(r[c_])).font = BLUE
        E.cell(i, 15, f'=IF(COUNT(J{i}:N{i})=0,"",SUM(J{i}:N{i}))').font = AR
        E.cell(i, 16, f'=IF(O{i}="","Not appraised",IF(O{i}>=Summary!$C$4,"High",IF(O{i}>=Summary!$C$5,"Moderate","Low")))').font = AR
        for j, f in enumerate(F_COLS, 17): E.cell(i, j, val(r[f])).font = BLUE
        E.cell(i, 32, val(r['Null_or_mixed_finding'])).font = AR; E.cell(i, 33, r['Rule_notes']).font = AR
    for j in range(1, len(hdr) + 1):
        E.column_dimensions[L(j)].width = {1: 8, 2: 11, 3: 9, 4: 44, 5: 24, 6: 30, 7: 16, 8: 34, 9: 7}.get(j, 11 if j < 33 else 45)
    E.row_dimensions[1].height = 45; E.freeze_panes = 'E2'; E.auto_filter.ref = f'A1:{L(len(hdr))}{last}'

    tier_rng, st_rng = f'Appendix_E!$P${first}:$P${last}', f'Appendix_E!$B${first}:$B${last}'
    S['A1'] = 'Evidence-quality appraisal: summary (live formulas on Appendix_E)'; S['A1'].font = Font(name='Arial', size=12, bold=True)
    S['A3'] = 'Settings (pre-specified; edit to re-run)'; S['A3'].font = AB
    for k, (lab, v) in enumerate([('High tier if score >=', HIGH_MIN), ('Moderate tier if score >=', MOD_MIN),
                                  ('Mechanism "Robust" if high-quality studies >=', ROBUST_MIN),
                                  ('Mechanism "Partial" if high-quality studies >=', PARTIAL_MIN)], 4):
        S.cell(k, 1, lab).font = AR; S.cell(k, 3, v).font = BLUE
    S['A9'] = 'Table S1. Quality tier by technology stream'; S['A9'].font = AB
    for j, h in enumerate(['Stream', 'High', 'Moderate', 'Low', 'Not appraised', 'Total', 'Appraised', '% High', '% Moderate', '% Low', 'Mean score'], 1):
        c = S.cell(10, j, h); c.font = AB; c.fill = HDR
    for k, st in enumerate(['Analytics', 'Blockchain', 'Integrated'], 11):
        S.cell(k, 1, st).font = AR
        for j, t in enumerate(['High', 'Moderate', 'Low', 'Not appraised'], 2):
            S.cell(k, j, f'=COUNTIFS({st_rng},$A{k},{tier_rng},"{t}")').font = AR
        S.cell(k, 6, f'=SUM(B{k}:E{k})').font = AR; S.cell(k, 7, f'=B{k}+C{k}+D{k}').font = AR
        for j, col in zip((8, 9, 10), 'BCD'):
            c = S.cell(k, j, f'=IFERROR({col}{k}/$G{k},0)'); c.number_format = '0.0%'; c.font = AR
        c = S.cell(k, 11, f'=IFERROR(AVERAGEIFS(Appendix_E!$O${first}:$O${last},{st_rng},$A{k}),"")'); c.number_format = '0.00'
    S.cell(14, 1, 'Total').font = AB
    for j in range(2, 8): S.cell(14, j, f'=SUM({L(j)}11:{L(j)}13)').font = AB
    for j, col in zip((8, 9, 10), 'BCD'):
        c = S.cell(14, j, f'=IFERROR({col}14/$G14,0)'); c.number_format = '0.0%'; c.font = AB
    c = S.cell(14, 11, f'=IFERROR(AVERAGE(Appendix_E!$O${first}:$O${last}),"")'); c.number_format = '0.00'; c.font = AB
    S['A17'] = 'Table S2. Study design profile'; S['A17'].font = AB
    for j, h in enumerate(['Design', 'Studies', '% of appraised'], 1):
        c = S.cell(18, j, h); c.font = AB; c.fill = HDR
    for k, dn in enumerate(DESIGNS, 19):
        S.cell(k, 1, dn).font = AR; S.cell(k, 2, f'=COUNTIF(Appendix_E!$H${first}:$H${last},A{k})').font = AR
        c = S.cell(k, 3, f'=IFERROR(B{k}/$G$14,0)'); c.number_format = '0.0%'
    S['A28'] = 'Table S3. Share of appraised studies meeting each criterion'; S['A28'].font = AB
    for j, h in enumerate(['Criterion', 'Met (%)'], 1):
        c = S.cell(29, j, h); c.font = AB; c.fill = HDR
    for k, (col, lab) in enumerate(zip('JKLMN', ['Q1 Design strength', 'Q2 Sample adequacy', 'Q3 Measurement / data credibility', 'Q4 Bias control', 'Q5 Analytical rigour']), 30):
        S.cell(k, 1, lab).font = AR; c = S.cell(k, 2, f'=AVERAGE(Appendix_E!${col}${first}:${col}${last})'); c.number_format = '0.0%'
    S.column_dimensions['A'].width = 52
    for j in range(2, 12): S.column_dimensions[L(j)].width = 13

    M = wb.create_sheet('Table_3b')
    M['A1'] = 'Table 3b. Mechanism support by evidence quality (appraised studies that examine the mechanism)'; M['A1'].font = Font(name='Arial', size=12, bold=True)
    for j, h in enumerate(['Factor', 'Mechanism', 'All studies', 'High quality', 'Moderate', 'Low', 'Support: pre-specified (high only)',
                           'High + Moderate', 'Support: sensitivity (high + moderate)', '% in High + Moderate'], 1):
        c = M.cell(3, j, h); c.font = AB; c.fill = HDR; c.alignment = Alignment(wrap_text=True, vertical='center')
    for k, f in enumerate(MECH, 4):
        col = L(17 + F_COLS.index(f)); rng = f'Appendix_E!${col}${first}:${col}${last}'
        M.cell(k, 1, f).font = AR; M.cell(k, 2, MECH[f][0]).font = AR
        M.cell(k, 3, f'=COUNTIFS({rng},1,{tier_rng},"<>Not appraised")').font = AR
        for j, t in zip((4, 5, 6), ('High', 'Moderate', 'Low')): M.cell(k, j, f'=COUNTIFS({rng},1,{tier_rng},"{t}")').font = AR
        M.cell(k, 7, f'=IF(D{k}>=Summary!$C$6,"Robust",IF(D{k}>=Summary!$C$7,"Partial","Weak"))').font = AR
        M.cell(k, 8, f'=D{k}+E{k}').font = AR
        M.cell(k, 9, f'=IF(H{k}>=Summary!$C$6,"Robust",IF(H{k}>=Summary!$C$7,"Partial","Weak"))').font = AR
        c = M.cell(k, 10, f'=IFERROR(H{k}/C{k},0)'); c.number_format = '0.0%'
    M['A20'] = ('Note. A study is counted when the mechanism appears in its extracted inputs, mediators/moderators, outputs, key findings or resilience '
                'dimension (keyword coding). Counts show that a study examines the mechanism, not that its effect was significant or positive.')
    M['A20'].font = Font(name='Arial', size=9, italic=True); M['A20'].alignment = Alignment(wrap_text=True, vertical='top'); M.merge_cells('A20:J22')
    M.column_dimensions['A'].width = 8; M.column_dimensions['B'].width = 30
    for j in range(3, 11): M.column_dimensions[L(j)].width = 17
    M.row_dimensions[3].height = 55
    wb.save(path)

build_workbook(d, OUT_DIR / 'Appendix_E_Quality_Appraisal.xlsx')
print('Saved:', *[p.name for p in sorted(OUT_DIR.iterdir())], sep='\n  ')

Saved:
  Appendix_E_Quality_Appraisal.xlsx
  appendix_E_study_level.csv
  circularity_check.csv
  data_checks.csv
  fig_evidence_quality.png
  manual_overrides_template.csv
  second_coder_template.csv
  table_3b.csv
  threshold_sweep.csv
